# 7장: 채팅 애플리케이션 구축
## Microsoft Foundry Models API 빠른 시작

이 노트북은 [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst)에서 가져온 것으로, [Azure OpenAI](notebook-azure-openai.ipynb) 서비스에 접근하는 노트북을 포함합니다.

> **참고:** GitHub Models는 2026년 7월 말에 종료됩니다. 이 노트북은 현재 동일한 무료 체험 모델 카탈로그와 Azure AI 추론 SDK 경험을 제공하는 [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst)를 대상으로 합니다.


# 개요  
"대형 언어 모델은 텍스트를 텍스트로 매핑하는 함수입니다. 입력된 텍스트 문자열이 주어지면, 대형 언어 모델은 다음에 올 텍스트를 예측하려고 합니다"(1). 이 "빠른 시작" 노트북은 사용자에게 고수준 LLM 개념, AML 시작을 위한 핵심 패키지 요구 사항, 프롬프트 설계에 대한 부드러운 소개, 그리고 여러 가지 다양한 사용 사례에 대한 짧은 예제를 소개합니다. 


## 목차  

[개요](#overview)  
[OpenAI 서비스 사용 방법](#how-to-use-openai-service)  
[1. OpenAI 서비스 생성하기](#1.-creating-your-openai-service)  
[2. 설치](#2.-installation)    
[3. 자격 증명](#3.-credentials)  

[사용 사례](#use-cases)    
[1. 텍스트 요약](#1.-summarize-text)  
[2. 텍스트 분류](#2.-classify-text)  
[3. 새로운 제품명 생성](#3.-generate-new-product-names)  
[4. 분류기 미세 조정](#4.fine-tune-a-classifier)  

[참고 문헌](#references)


### 첫 번째 프롬프트 만들기  
이 짧은 연습은 Microsoft Foundry Models에서 간단한 작업인 "요약"을 위한 모델에 프롬프트를 제출하는 기본적인 소개를 제공합니다.


<strong>단계</strong>:  
1. python 환경에 `azure-ai-inference` 라이브러리를 설치하세요(아직 설치하지 않았다면).  
2. 표준 헬퍼 라이브러리를 불러오고 Microsoft Foundry Models 자격 증명을 설정하세요.  
3. 작업에 맞는 모델을 선택하세요  
4. 모델을 위한 간단한 프롬프트를 만드세요  
5. 모델 API에 요청을 제출하세요!


### 1. `azure-ai-inference` 설치하기


In [ ]:
%pip install azure-ai-inference

### 2. 헬퍼 라이브러리 가져오기 및 자격 증명 인스턴스화


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. 적합한 모델 찾기  
GPT-4o 및 GPT-4o mini와 같은 모델은 자연어를 이해하고 생성할 수 있으며, Meta, Mistral, Cohere, Microsoft의 모델과 함께 Microsoft Foundry Models 카탈로그에서 사용할 수 있습니다.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. 프롬프트 디자인  

"대규모 언어 모델의 마법은 방대한 양의 텍스트에 대해 이 예측 오류를 최소화하도록 학습됨으로써 모델이 이러한 예측에 유용한 개념을 학습하게 된다는 점입니다. 예를 들어 그들은 다음과 같은 개념을 학습합니다"(1):

* 맞춤법 방법
* 문법 작동 방식
* 바꾸어 표현하는 방법
* 질문에 답하는 방법
* 대화를 나누는 방법
* 여러 언어로 글쓰는 방법
* 코딩하는 방법
* 기타 등등

#### 대규모 언어 모델을 제어하는 방법  
"대규모 언어 모델에 입력되는 모든 것 중에서 가장 영향력 있는 것은 단연 텍스트 프롬프트입니다(1).

대규모 언어 모델은 몇 가지 방법으로 출력 생성을 유도할 수 있습니다:

지시: 모델에게 원하는 것을 알려줍니다
완성: 원하는 시작 부분을 모델이 완성하도록 유도합니다
시범: 모델에게 원하는 것을 보여줍니다, 다음과 같이:
프롬프트 내 몇 가지 예시
미세 조정 학습 데이터셋에 수백 또는 수천 개의 예시"



#### 프롬프트 생성에 대한 세 가지 기본 지침:

**보여주고 말하기**. 지시, 예시 또는 둘의 조합을 통해 원하는 바를 명확히 하십시오. 모델에게 항목 목록을 알파벳 순으로 정렬하거나 단락을 감정별로 분류하게 하려면, 그것이 원하는 바임을 보여주십시오.

**양질의 데이터를 제공하기**. 분류기를 만들거나 모델이 패턴을 따르도록 하려면 충분한 예제가 있는지 확인하십시오. 예제를 교정하는 것을 잊지 마십시오 — 모델은 보통 기본적인 철자 실수를 보고 응답할 만큼 똑똑하지만, 이것을 의도한 것으로 오해할 수 있으며 응답에 영향을 줄 수 있습니다.

**설정 확인하기.** 온도(temperature)와 top_p 설정은 모델이 응답을 생성할 때 얼마나 결정론적인지를 제어합니다. 답이 하나뿐인 응답을 요청할 경우 이 값을 낮게 설정하는 것이 좋습니다. 더 다양한 응답을 원한다면 값을 높게 설정할 수 있습니다. 사람들이 이 설정에 관해 가장 많이 하는 실수는 이것을 '영리함'이나 '창의성' 제어라고 생각하는 것입니다.


출처: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. 제출!  


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### 같은 호출을 반복하면 결과가 어떻게 비교됩니까?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## 텍스트 요약  
#### 도전 과제  
텍스트 구절 끝에 'tl;dr:'을 추가하여 텍스트를 요약하세요. 모델이 추가 지시 없이 여러 작업을 수행하는 방법을 이해하는 방식을 주목하세요. tl;dr보다 더 설명적인 프롬프트로 실험하여 모델의 동작을 수정하고 받는 요약을 사용자 맞춤화할 수 있습니다(3).  

최근 연구는 대량의 텍스트 코퍼스에 대한 사전 훈련과 그 후 특정 작업에 대한 미세 조정을 통해 많은 NLP 작업과 벤치마크에서 상당한 성과 향상을 보여주었습니다. 일반적으로 아키텍처는 작업에 무관하지만, 이 방법은 여전히 수천 또는 수만 개의 예제로 된 작업별 미세 조정 데이터 세트를 필요로 합니다. 반면, 인간은 일반적으로 몇 가지 예시나 간단한 지시만으로 새로운 언어 작업을 수행할 수 있는데, 이는 현재 NLP 시스템이 여전히 크게 어려움을 겪고 있는 부분입니다. 여기서는 언어 모델의 규모를 확장하면 작업에 무관한 소수 샷 성능이 크게 향상되어 때로는 이전 최첨단 미세 조정 방법과 경쟁할 만한 수준에 도달함을 보여줍니다.  



요약  


# 여러 사용 사례를 위한 연습  
1. 텍스트 요약  
2. 텍스트 분류  
3. 새로운 제품 이름 생성


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## 텍스트 분류  
#### 도전 과제  
추론 시 제공된 카테고리로 항목을 분류합니다. 다음 예시에서는 프롬프트(*playground_reference) 내에 분류할 카테고리와 텍스트를 모두 제공합니다. 

고객 문의: 안녕하세요, 제 노트북 키보드의 키 중 하나가 최근에 고장나서 교체가 필요합니다:

분류된 카테고리:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## 새로운 제품 이름 생성
#### 도전 과제
예시 단어에서 제품 이름을 만듭니다. 여기에는 생성할 제품에 관한 정보를 프롬프트에 포함합니다. 또한 원하는 패턴을 보여주기 위해 유사한 예시도 제공합니다. 무작위성과 혁신적인 응답을 높이기 위해 온도 값도 높게 설정했습니다.

제품 설명: 가정용 밀크셰이크 제조기
시드 단어: 빠른, 건강한, 컴팩트한.
제품 이름: HomeShaker, Fit Shaker, QuickShake, Shake Maker

제품 설명: 모든 발 크기에 맞는 신발 한 켤레
시드 단어: 적응형, 맞춤형, 옴니핏.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# 참고문헌  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [텍스트 분류를 위한 GPT 모델 미세 조정 모범 사례](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# 추가 지원  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# 기여자
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
